# 🧪 AC3 – Loja Pokémon | Testes Automatizados Completos
**UNIFECAF – Professor Rodrigo Moreira**

Grupo: Vinnicius, Matheus, Willian, Gabriel

---
## 📋 Cobertura de Testes

| Categoria | Quantidade | Descrição |
|---|---|---|
| ✅ Positivos | 5 | Comportamentos esperados e corretos da API |
| ❌ Negativos | 5 | Entradas inválidas, erros e limites da API |
| **Total** | **10** | **Testes de API Pokémon TCG** |

---
> Execute com **Ctrl+F9** (Executar tudo) ou célula por célula com **Shift+Enter**

## 📦 Célula 1 — Instalação e Configuração

In [ ]:
!pip install requests matplotlib -q
import requests, time, matplotlib.pyplot as plt

BASE = 'https://api.pokemontcg.io/v2'
GITHUB_PAGES = 'https://willzao03.github.io/Loja-Pokemon-projeto/'
GITHUB_REPO  = 'https://github.com/willzao03/Loja-Pokemon-projeto'

resultados = []

def log(ok, nome, detalhe=''):
    icone = '\033[92m✅ PASSOU\033[0m' if ok else '\033[91m❌ FALHOU\033[0m'
    print(f'{icone} | {nome}')
    if detalhe: print(f'         ↳ {detalhe}')
    resultados.append({'ok': ok, 'nome': nome, 'detalhe': detalhe})

def get(url, timeout=25):
    try:
        return requests.get(url, timeout=timeout)
    except Exception as e:
        print(f'         ⚠️  Erro: {type(e).__name__}')
        return None

print('✅ Ambiente pronto! API:', BASE)

---
# ✅ TESTES POSITIVOS
> Verificam que a API funciona corretamente em cenários válidos.

## POS-01 — Busca válida retorna status 200 e dados
**Objetivo:** Confirmar que a API responde com sucesso ao buscar 'Charizard'

**Critério:** `status == 200` e `len(data) > 0`

In [ ]:
url = f'{BASE}/cards?q=name:Charizard&pageSize=1'
print(f'🔗 {url}\n')
t0 = time.time()
r = get(url)
tempo = time.time() - t0
if r is None:
    log(False, 'POS-01 | Busca Charizard → status 200 + dados', 'Sem resposta da API')
else:
    data = r.json().get('data', [])
    ok = r.status_code == 200 and len(data) > 0
    log(ok, 'POS-01 | Busca Charizard → status 200 + dados',
        f'Status: {r.status_code} | Cartas: {len(data)} | Tempo: {tempo:.2f}s')
    if data:
        c = data[0]
        print(f'\n   🃏 Carta: {c.get("name")} | Set: {c.get("set",{}).get("name","?")} | ID: {c.get("id")}')

## POS-02 — Campos obrigatórios presentes na resposta
**Objetivo:** Garantir que cada carta retornada tem `id`, `name` e `images`

**Critério:** Todos os campos obrigatórios existem no objeto retornado

In [ ]:
url = f'{BASE}/cards?q=name:Mewtwo&pageSize=1'
print(f'🔗 {url}\n')
r = get(url)
if r is None:
    log(False, 'POS-02 | Campos obrigatórios (id, name, images)', 'Sem resposta')
else:
    data = r.json().get('data', [])
    card = data[0] if data else {}
    esperados = ['id', 'name', 'images']
    presentes = [f for f in esperados if f in card]
    ausentes  = [f for f in esperados if f not in card]
    ok = len(ausentes) == 0
    log(ok, 'POS-02 | Campos obrigatórios (id, name, images)',
        f'Presentes: {presentes} | Ausentes: {ausentes}')
    print(f'\n   📋 Todos os campos retornados ({len(card)}):')
    for k in list(card.keys())[:10]:
        print(f'      {k}: {str(card[k])[:50]}')

## POS-03 — Paginação pageSize=3 respeitada
**Objetivo:** Verificar que o parâmetro `pageSize` limita corretamente os resultados

**Critério:** `status == 200` e `len(data) <= 3`

In [ ]:
url = f'{BASE}/cards?q=name:Pikachu&pageSize=3'
print(f'🔗 {url}\n')
r = get(url)
if r is None:
    log(False, 'POS-03 | Paginação pageSize=3 respeitada', 'Sem resposta')
else:
    data = r.json().get('data', [])
    total_count = r.json().get('totalCount', '?')
    ok = r.status_code == 200 and len(data) <= 3
    log(ok, 'POS-03 | Paginação pageSize=3 respeitada',
        f'Status: {r.status_code} | Retornados: {len(data)} | Total na API: {total_count}')
    print(f'\n   📄 Cartas retornadas:')
    for c in data:
        print(f'      • {c.get("name")} [{c.get("id")}]')

## POS-04 — Content-Type da resposta é application/json
**Objetivo:** Confirmar que a API retorna o header correto

**Critério:** `Content-Type` contém `application/json`

In [ ]:
url = f'{BASE}/cards?q=name:Pikachu&pageSize=1'
print(f'🔗 {url}\n')
r = get(url)
if r is None:
    log(False, 'POS-04 | Content-Type é application/json', 'Sem resposta')
else:
    ct = r.headers.get('Content-Type', '')
    ok = 'application/json' in ct
    log(ok, 'POS-04 | Content-Type é application/json', f'Content-Type: {ct}')
    print(f'\n   📋 Headers relevantes:')
    for h in ['Content-Type', 'X-Ratelimit-Limit', 'X-Ratelimit-Remaining']:
        print(f'      {h}: {r.headers.get(h, "não presente")}')

## POS-05 — Tempo de resposta menor que 5 segundos
**Objetivo:** Garantir performance aceitável da API

**Critério:** Tempo de resposta `< 5.0s`

In [ ]:
url = f'{BASE}/cards?q=name:Blastoise&pageSize=1'
print(f'🔗 {url}\n')
tempos = []
print('   Executando 3 medições...')
for i in range(3):
    t0 = time.time()
    r = get(url)
    t = time.time() - t0
    tempos.append(t)
    status = r.status_code if r else 'ERR'
    print(f'   Medição {i+1}: {t:.2f}s | Status: {status}')
media = sum(tempos)/len(tempos)
ok = media < 5.0
log(ok, 'POS-05 | Tempo de resposta < 5s',
    f'Média: {media:.2f}s | Min: {min(tempos):.2f}s | Max: {max(tempos):.2f}s')

---
# ❌ TESTES NEGATIVOS
> Verificam o comportamento da API em cenários inválidos ou de erro.

## NEG-01 — Carta inexistente retorna lista vazia
**Objetivo:** Confirmar que busca por nome inexistente retorna `data: []` e não um erro

**Critério:** `status == 200` e `data == []`

In [ ]:
url = f'{BASE}/cards?q=name:CARTAINEXISTENTE999XYZ'
print(f'🔗 {url}\n')
r = get(url)
if r is None:
    log(False, 'NEG-01 | Carta inexistente → data vazio []', 'Sem resposta')
else:
    data = r.json().get('data', None)
    ok = r.status_code == 200 and isinstance(data, list) and len(data) == 0
    log(ok, 'NEG-01 | Carta inexistente → data vazio []',
        f'Status: {r.status_code} | data: {data} | totalCount: {r.json().get("totalCount","?")}')
    print(f'\n   💡 Comportamento correto: API retorna 200 com lista vazia')
    print(f'      (não retorna 404, pois a requisição foi válida)')

## NEG-02 — Endpoint inválido retorna 404
**Objetivo:** Confirmar que rota inexistente retorna erro 404

**Critério:** `status == 404`

In [ ]:
url = f'{BASE}/rotainexistente'
print(f'🔗 {url}\n')
r = get(url)
if r is None:
    log(False, 'NEG-02 | Endpoint inválido → 404', 'Sem resposta')
else:
    ok = r.status_code == 404
    log(ok, 'NEG-02 | Endpoint inválido → 404', f'Status: {r.status_code}')
    try:
        body = r.json()
        print(f'\n   📋 Corpo da resposta: {body}')
    except:
        print(f'\n   📋 Resposta (texto): {r.text[:100]}')

## NEG-03 — pageSize=0 não deve retornar dados ⚠️ BUG DA API
**Objetivo:** Verificar comportamento com parâmetro inválido `pageSize=0`

**Critério esperado:** `status >= 400` ou `data == []`

> ⚠️ **Resultado real:** A API ignorou o parâmetro e retornou dados normalmente — bug da API externa.

In [ ]:
url = f'{BASE}/cards?q=name:Pikachu&pageSize=0'
print(f'🔗 {url}\n')
r = get(url)
if r is None:
    log(False, 'NEG-03 | pageSize=0 não retorna dados', 'Sem resposta')
else:
    data = r.json().get('data', [])
    ok = r.status_code >= 400 or len(data) == 0
    log(ok, 'NEG-03 | pageSize=0 não retorna dados',
        f'Status: {r.status_code} | Itens retornados: {len(data)}')
    print()
    if not ok:
        print('   ⚠️  ANÁLISE: Bug confirmado na API pokemontcg.io')
        print(f'      pageSize=0 deveria retornar erro, mas retornou {len(data)} itens')
        print('   📌 Impacto na Loja: nenhum — a loja não usa pageSize=0')
        print('   📌 Recomendação: validar pageSize >= 1 antes de enviar à API')

## NEG-04 — Filtro por campo inexistente não retorna dados
**Objetivo:** Verificar que filtros com campos inválidos não retornam resultados

**Critério:** `status >= 400` ou `data == []`

In [ ]:
url = f'{BASE}/cards?q=campofalso:valorfalso'
print(f'🔗 {url}\n')
r = get(url)
if r is None:
    log(False, 'NEG-04 | Campo inexistente → sem dados', 'Sem resposta')
else:
    data = r.json().get('data', [])
    ok = r.status_code >= 400 or len(data) == 0
    log(ok, 'NEG-04 | Campo inexistente → sem dados',
        f'Status: {r.status_code} | Itens: {len(data)}')
    print(f'\n   💡 A API tratou o campo desconhecido sem retornar dados')

## NEG-05 — GitHub Pages: disponibilidade do deploy
**Objetivo:** Verificar se o projeto está publicado no GitHub Pages

**Critério:** `status == 200` (deploy ativo)

In [ ]:
print(f'🔗 {GITHUB_PAGES}\n')
r = get(GITHUB_PAGES, timeout=15)
if r is None:
    log(False, 'NEG-05 | GitHub Pages INATIVO', 'Sem resposta — ativar em Settings > Pages')
elif r.status_code == 200:
    log(True, 'NEG-05 | GitHub Pages ATIVO (deploy realizado)',
        f'Status: {r.status_code} | URL: {GITHUB_PAGES}')
    print(f'\n   ✅ Projeto publicado e acessível publicamente')
else:
    log(False, 'NEG-05 | GitHub Pages com status inesperado',
        f'Status: {r.status_code} — Verificar configuração do Pages')

---
# 📊 Resumo Final e Gráfico

In [ ]:
passou = sum(1 for r in resultados if r['ok'])
falhou = len(resultados) - passou
print('\n' + '='*60)
print('  RESULTADO FINAL — AC3 Loja Pokémon')
print('='*60)
print(f'  {'Teste':<45} {'Resultado'}')
print('-'*60)
for r in resultados:
    icone = '✅' if r['ok'] else '❌'
    print(f'  {icone} {r["nome"][:50]}')
    if r['detalhe']:
        print(f'     ↳ {r["detalhe"][:55]}')
print('='*60)
print(f'  Passaram : {passou}/{len(resultados)}')
print(f'  Falharam : {falhou}/{len(resultados)}')
print('='*60)

In [ ]:
# Gráfico de pizza
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#1a1a2e')

# Pizza
ax1 = axes[0]
ax1.set_facecolor('#16213e')
cores = ['#00e676', '#e3350d']
vals  = [passou, falhou] if falhou > 0 else [passou]
labs  = [f'Passou ({passou})', f'Falhou ({falhou})'] if falhou > 0 else [f'Passou ({passou})']
cs    = cores if falhou > 0 else [cores[0]]
wedges, texts, autotexts = ax1.pie(vals, labels=labs, colors=cs,
    autopct='%1.0f%%', startangle=90,
    textprops={'color': 'white', 'fontsize': 12})
for at in autotexts: at.set_color('white'); at.set_fontsize(13)
ax1.set_title('Resultado Geral', color='#ffcb05', fontsize=14, pad=15)

# Barras por teste
ax2 = axes[1]
ax2.set_facecolor('#16213e')
nomes = [r['nome'].split('|')[0].strip() for r in resultados]
cores_bar = ['#00e676' if r['ok'] else '#e3350d' for r in resultados]
bars = ax2.barh(nomes, [1]*len(resultados), color=cores_bar, edgecolor='#0f3460')
ax2.set_xlim(0, 1.3)
ax2.set_xticks([])
for i, r in enumerate(resultados):
    ax2.text(1.05, i, '✅ PASSOU' if r['ok'] else '❌ FALHOU',
             va='center', color='white', fontsize=9)
ax2.tick_params(colors='white')
for spine in ax2.spines.values(): spine.set_edgecolor('#0f3460')
ax2.set_title('Resultado por Teste', color='#ffcb05', fontsize=14, pad=15)
plt.setp(ax2.get_yticklabels(), color='white', fontsize=8)

plt.suptitle('AC3 – Loja Pokémon | Testes de API', color='#ffcb05',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('resultado_testes.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('\n📁 Gráfico salvo como resultado_testes.png')

---
# 📝 Análise de Falhas
> Documentação técnica dos testes que não passaram.

In [ ]:
falhas = [r for r in resultados if not r['ok']]
if not falhas:
    print('🎉 Todos os testes passaram nesta execução!')
else:
    print(f'⚠️  {len(falhas)} teste(s) com falha:\n')
    analises = {
        'POS-01': ('Timeout da API externa', 'Instabilidade de rede ou rate limit da pokemontcg.io', 'Reexecutar em outro momento'),
        'POS-02': ('Timeout da API externa', 'Instabilidade de rede ou rate limit da pokemontcg.io', 'Reexecutar em outro momento'),
        'NEG-03': ('Bug na API externa', 'pokemontcg.io ignora pageSize=0 e retorna dados', 'Validar pageSize >= 1 no frontend'),
        'NEG-05': ('Deploy não realizado', 'GitHub Pages não está ativo no repositório', 'Ativar em Settings > Pages > Branch: main'),
    }
    for r in falhas:
        codigo = r['nome'].split('|')[0].strip()
        info = analises.get(codigo, ('Falha inesperada', r['detalhe'], 'Verificar manualmente'))
        print(f'  ❌ {r["nome"]}')
        print(f'     Causa    : {info[0]}')
        print(f'     Detalhe  : {info[1]}')
        print(f'     Ação     : {info[2]}')
        print()